# 01 — Data Preparation and Processing

## Human-to-AI Interaction Shift Analysis

> This notebook prepares the raw datasets used in the DSA 210 project.  
> The main goal is to clean, organize, and transform the data before running exploratory data analysis and hypothesis testing.

---

## Project Focus

This project examines whether people are shifting from traditional human-based help-seeking platforms toward AI-based platforms.

The cleaned datasets created in this notebook will be used in the next stage for:

- comparing search trends between AI-related and traditional platforms,
- analyzing recent platform-level traffic patterns,
- supporting the findings with external survey data,
- conducting hypothesis tests on the observed trends.

---

### Data Sources

| Data Type | Purpose |
|---|---|
| **Google Trends keyword data** | Measures search interest for AI and traditional help-seeking keywords |
| **Platform traffic data** | Provides recent platform-level traffic context |
| **Stack Overflow Survey** | Provides survey-based evidence on AI tool adoption among technical respondents |
| **Pew AI Survey** | Provides broader public opinion and AI adoption context |

---

### Folder Structure

`data/raw/`  
Original input files collected from external sources.

`data/processed/`  
Cleaned and analysis-ready files produced by this notebook.

## 1. Setup and File Path Configuration

This section imports the required libraries and defines the file paths used throughout the notebook. These paths allow the notebook to read raw datasets and save cleaned outputs in a consistent way.


**Code purpose:** Import the required libraries and define the main project folders.

In [70]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_DIR = Path.home() / "Desktop" / "dsa"

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project folder:", PROJECT_DIR)
print("Raw folder exists:", RAW_DIR.exists())
print("Processed folder exists:", PROCESSED_DIR.exists())

Project folder: /Users/beren/Desktop/dsa
Raw folder exists: True
Processed folder exists: True


## 2. Checking Available Data Files

Before processing the datasets, I check the files inside the raw and processed folders. This confirms that the project files are organized correctly and ready for cleaning.

**Code purpose:** Check whether the expected raw and processed data files are available.

In [71]:
print("Files in raw folder:")
for file in RAW_DIR.iterdir():
    if file.name != ".DS_Store":
        print("-", file.name)

print("\nFiles in processed folder:")
for file in PROCESSED_DIR.iterdir():
    if file.name != ".DS_Store":
        print("-", file.name)

Files in raw folder:
- stackoverflow_survey_2023.csv
- stackoverflow_survey_2024.csv
- stackoverflow_survey_2025.csv
- pew_global_ai_survey_2025.csv
- schemas
- Study Support Keyword Comparison.csv
- General Help-Seeking Keyword Comparison.csv
- Advice-Seeking Keyword Comparison.csv

Files in processed folder:
- stackoverflow_ai_adoption_summary.csv
- pew_ai_survey_cleaned.csv
- stackoverflow_ai_usage_cleaned.csv
- platform_traffic_context.csv
- stackoverflow_ai_adoption_valid_summary.csv
- google_trends_cleaned.csv


## 3. Cleaning Google Trends Data

This section cleans the Google Trends keyword comparison files. The original files are converted into a consistent long-format dataset so that different keywords, categories, and platform types can be compared more easily.

The cleaned output is saved as `google_trends_cleaned.csv` in the processed data folder.

**Code purpose:** Load the Google Trends files, clean their structure, and convert them into a long-format dataset.

In [72]:
trend_files = {
    "advice_seeking": "Advice-Seeking Keyword Comparison.csv",
    "general_help_seeking": "General Help-Seeking Keyword Comparison.csv",
    "study_support": "Study Support Keyword Comparison.csv"
}

all_trends = []

for category, filename in trend_files.items():
    file_path = RAW_DIR / filename
    
    # Find the row that contains the real column names
    preview = pd.read_csv(file_path, nrows=10, header=None)
    
    header_row = None
    for i in range(len(preview)):
        first_cell = str(preview.iloc[i, 0]).lower()
        if first_cell in ["week", "month", "time"]:
            header_row = i
            break
    
    df = pd.read_csv(file_path, skiprows=header_row)
    
    date_col = df.columns[0]
    df = df.rename(columns={date_col: "date"})
    
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])
    
    long_df = df.melt(
        id_vars="date",
        var_name="keyword",
        value_name="trend_score"
    )
    
    long_df["keyword"] = (
        long_df["keyword"]
        .str.replace(": \\(Worldwide\\)", "", regex=True)
        .str.strip()
    )
    
    long_df["trend_score"] = (
        long_df["trend_score"]
        .astype(str)
        .str.replace("<1", "0.5")
    )
    
    long_df["trend_score"] = pd.to_numeric(long_df["trend_score"], errors="coerce")
    long_df = long_df.dropna(subset=["trend_score"])
    
    long_df["category"] = category
    all_trends.append(long_df)

google_trends_cleaned = pd.concat(all_trends, ignore_index=True)

google_trends_cleaned.head()

,date,keyword,trend_score,category
0,2021-04-01,Yahoo Answers,56.0,advice_seeking
1,2021-05-01,Yahoo Answers,22.0,advice_seeking
2,2021-06-01,Yahoo Answers,14.0,advice_seeking
3,2021-07-01,Yahoo Answers,12.0,advice_seeking
4,2021-08-01,Yahoo Answers,11.0,advice_seeking


## 4. Classifying Keywords by Platform Type

After cleaning the Google Trends files, I classify each keyword by platform type. This allows the project to compare AI-based platforms with traditional Q&A, forum, and study-support platforms.

**Code purpose:** Classify each keyword as an AI platform, forum/Q&A platform, study-support platform, or traditional platform.

In [73]:
ai_keywords = ["chatgpt", "claude", "gemini", "perplexity", "character ai", "ai friend"]

def classify_platform_type(keyword):
    keyword_lower = str(keyword).lower()

    if any(ai_word in keyword_lower for ai_word in ai_keywords):
        return "AI platform"
    elif keyword_lower in ["yahoo answers", "ask.fm", "quora", "reddit"]:
        return "forum / Q&A platform"
    elif keyword_lower in ["chegg", "course hero", "quizlet", "khan academy"]:
        return "study support platform"
    else:
        return "traditional platform"

google_trends_cleaned["platform_type"] = google_trends_cleaned["keyword"].apply(classify_platform_type)

google_trends_cleaned = google_trends_cleaned[
    ["date", "category", "keyword", "platform_type", "trend_score"]
]

google_trends_cleaned.to_csv(PROCESSED_DIR / "google_trends_cleaned.csv", index=False)

print("Saved file:", PROCESSED_DIR / "google_trends_cleaned.csv")
print("Shape:", google_trends_cleaned.shape)

google_trends_cleaned.head()

Saved file: /Users/beren/Desktop/dsa/data/processed/google_trends_cleaned.csv
Shape: (1091, 5)


,date,category,keyword,platform_type,trend_score
0,2021-04-01,advice_seeking,Yahoo Answers,forum / Q&A platform,56.0
1,2021-05-01,advice_seeking,Yahoo Answers,forum / Q&A platform,22.0
2,2021-06-01,advice_seeking,Yahoo Answers,forum / Q&A platform,14.0
3,2021-07-01,advice_seeking,Yahoo Answers,forum / Q&A platform,12.0
4,2021-08-01,advice_seeking,Yahoo Answers,forum / Q&A platform,11.0


**Code purpose:** Check whether each keyword was assigned to the correct category and platform type.

In [74]:
google_trends_cleaned.groupby(["category", "platform_type", "keyword"]).size()

category              platform_type           keyword      
advice_seeking        AI platform             ai friend         61
                      forum / Q&A platform    Ask.fm            61
                                              Yahoo Answers     61
general_help_seeking  AI platform             chatgpt help      61
                      forum / Q&A platform    quora             61
study_support         AI platform             chatgpt study    262
                      study support platform  chegg            262
                                              course hero      262
dtype: int64

## 5. Preparing Platform Traffic Context Data

This section loads the platform traffic dataset, which includes recent visit values for selected AI, Q&A, and study-support platforms.

Since this dataset covers a shorter period, it is used as supporting context rather than the main evidence for long-term behavioral change.

**Code purpose:** Load the platform traffic dataset and convert month and visit columns into usable data types.

In [75]:
traffic_file = PROCESSED_DIR / "platform_traffic_context.csv"

traffic_context = pd.read_csv(traffic_file)

traffic_context.columns = traffic_context.columns.str.strip().str.lower()

traffic_context["month"] = pd.to_datetime(traffic_context["month"], errors="coerce")
traffic_context["collection_date"] = pd.to_datetime(traffic_context["collection_date"], errors="coerce")

traffic_context["visits"] = (
    traffic_context["visits"]
    .astype(str)
    .str.replace(".", "", regex=False)
    .str.replace(",", "", regex=False)
)

traffic_context["visits"] = pd.to_numeric(traffic_context["visits"], errors="coerce")

print("Shape:", traffic_context.shape)
traffic_context.head()

Shape: (54, 9)


,platform,domain,platform_type,category,month,visits,metric,source,collection_date
0,ChatGPT,chatgpt.com,AI platform,general_help_seeking,2025-10-01,6170000000,visits,WebsiteTrafficChecker,2026-05-03
1,ChatGPT,chatgpt.com,AI platform,general_help_seeking,2025-11-01,5840000000,visits,WebsiteTrafficChecker,2026-05-03
2,ChatGPT,chatgpt.com,AI platform,general_help_seeking,2025-12-01,5520000000,visits,WebsiteTrafficChecker,2026-05-03
3,ChatGPT,chatgpt.com,AI platform,general_help_seeking,2026-01-01,5720000000,visits,WebsiteTrafficChecker,2026-05-03
4,ChatGPT,chatgpt.com,AI platform,general_help_seeking,2026-02-01,5350000000,visits,WebsiteTrafficChecker,2026-05-03


**Code purpose:** Confirm that each platform has six months of traffic data.

In [76]:
traffic_context.groupby(["platform_type", "platform"])["month"].nunique()

platform_type           platform    
AI platform             Character AI    6
                        ChatGPT         6
                        Claude          6
Forum Q&A platform      Quora           6
General help platform   WikiHow         6
Study support platform  Chegg           6
                        Course Hero     6
                        Khan Academy    6
                        Quizlet         6
Name: month, dtype: int64

**Code purpose:** Save the cleaned traffic dataset back into the processed data folder.

In [77]:
traffic_context.to_csv(PROCESSED_DIR / "platform_traffic_context.csv", index=False)

print("Traffic context file saved successfully.")
print("Final shape:", traffic_context.shape)

Traffic context file saved successfully.
Final shape: (54, 9)


### Traffic Data Check

The platform traffic context dataset was successfully loaded and cleaned. Each selected platform has six months of visit data, covering the period from October 2025 to March 2026.

This dataset will be used as supporting context in the analysis notebook.

## 6. Preparing Stack Overflow Survey Data

This section prepares the Stack Overflow Survey data for external validation. Since the original survey files are large, I only extract the AI-related columns needed for this project.

The cleaned output will be saved as `stackoverflow_ai_usage_cleaned.csv` in the processed data folder.

**Code purpose:** Inspect the Stack Overflow survey files and identify AI-related columns for each year.

In [78]:
stackoverflow_files = {
    2023: "stackoverflow_survey_2023.csv",
    2024: "stackoverflow_survey_2024.csv",
    2025: "stackoverflow_survey_2025.csv"
}

for year, filename in stackoverflow_files.items():
    file_path = RAW_DIR / filename
    
    if file_path.exists():
        temp = pd.read_csv(file_path, nrows=5)
        ai_columns = [col for col in temp.columns if "ai" in col.lower()]
        
        print(f"\n{year} AI-related columns:")
        print(ai_columns)
    else:
        print(f"{year} file not found:", filename)


2023 AI-related columns:
['MainBranch', 'AISearchHaveWorkedWith', 'AISearchWantToWorkWith', 'AIDevHaveWorkedWith', 'AIDevWantToWorkWith', 'SOAI', 'AISelect', 'AISent', 'AIAcc', 'AIBen', 'AIToolInterested in Using', 'AIToolCurrently Using', 'AIToolNot interested in Using', 'AINextVery different', 'AINextNeither different nor similar', 'AINextSomewhat similar', 'AINextVery similar', 'AINextSomewhat different']

2024 AI-related columns:
['MainBranch', 'AISearchDevHaveWorkedWith', 'AISearchDevWantToWorkWith', 'AISearchDevAdmired', 'AISelect', 'AISent', 'AIBen', 'AIAcc', 'AIComplex', 'AIToolCurrently Using', 'AIToolInterested in Using', 'AIToolNot interested in Using', 'AINextMuch more integrated', 'AINextNo change', 'AINextMore integrated', 'AINextLess integrated', 'AINextMuch less integrated', 'AIThreat', 'AIEthics', 'AIChallenges']

2025 AI-related columns:
['MainBranch', 'LearnCodeAI', 'AILearnHow', 'AIThreat', 'AIModelsChoice', 'AIModelsHaveWorkedWith', 'AIModelsWantToWorkWith', 'AIMo

### Selecting AI-Related Survey Columns

The Stack Overflow survey includes many columns about AI tools. I selected only the columns that are most relevant to this project.

The selected columns show whether respondents use AI tools, how they feel about AI, how much they trust AI accuracy, and whether they see AI as beneficial. This keeps the processed dataset focused and easier to analyze.

**Code purpose:** Extract the AI-related Stack Overflow columns and save a smaller cleaned dataset.

In [79]:
selected_so_columns = [
    "MainBranch",
    "AISelect",
    "AISent",
    "AIAcc",
    "AIBen"
]

stackoverflow_cleaned_list = []

for year, filename in stackoverflow_files.items():
    file_path = RAW_DIR / filename
    
    df = pd.read_csv(file_path, low_memory=False)
    
    available_columns = [col for col in selected_so_columns if col in df.columns]
    
    cleaned = df[available_columns].copy()
    cleaned["year"] = year
    
    stackoverflow_cleaned_list.append(cleaned)

stackoverflow_ai_usage_cleaned = pd.concat(stackoverflow_cleaned_list, ignore_index=True)

stackoverflow_ai_usage_cleaned.to_csv(
    PROCESSED_DIR / "stackoverflow_ai_usage_cleaned.csv",
    index=False
)

print("Saved file:", PROCESSED_DIR / "stackoverflow_ai_usage_cleaned.csv")
print("Shape:", stackoverflow_ai_usage_cleaned.shape)

stackoverflow_ai_usage_cleaned.head()

Saved file: /Users/beren/Desktop/dsa/data/processed/stackoverflow_ai_usage_cleaned.csv
Shape: (203812, 6)


,MainBranch,AISelect,AISent,AIAcc,AIBen,year
0,None of these,NaN,NaN,NaN,NaN,2023
1,I am a developer by profession,Yes,Indifferent,Other (please explain),Somewhat distrust,2023
2,I am a developer by profession,"No, and I don't plan to",NaN,NaN,NaN,2023
3,I am a developer by profession,"No, and I don't plan to",NaN,NaN,NaN,2023
4,I am a developer by profession,Yes,Very favorable,Increase productivity;Greater efficiency;Speed...,Somewhat trust,2023


**Code purpose:** Check the number of survey responses included from each year.

In [80]:
stackoverflow_ai_usage_cleaned["year"].value_counts().sort_index()

year
2023    89184
2024    65437
2025    49191
Name: count, dtype: int64

**Code purpose:** Review how AI usage responses are distributed across survey years.

In [81]:
stackoverflow_ai_usage_cleaned.groupby("year")["AISelect"].value_counts(dropna=False)

year  AISelect                                   
2023  Yes                                            39042
      No, and I don't plan to                        26221
      No, but I plan to soon                         22710
      NaN                                             1211
2024  Yes                                            37662
      No, and I don't plan to                        14837
      No, but I plan to soon                          8408
      NaN                                             4530
2025  Yes, I use AI tools daily                      15883
      NaN                                            15471
      Yes, I use AI tools weekly                      5958
      No, and I don't plan to                         5454
      Yes, I use AI tools monthly or infrequently     4628
      No, but I plan to soon                          1797
Name: count, dtype: int64

### Creating a Stack Overflow AI Adoption Summary

After selecting the AI-related survey columns, I summarize the `AISelect` responses by year. Since the response options changed in 2025, I group them into broader categories: AI users, people planning to use AI, and people not planning to use AI.

**Code purpose:** Group AI usage responses into broader adoption categories and calculate yearly percentages.

In [82]:
def classify_ai_select(response):
    if pd.isna(response):
        return "missing"
    
    response = str(response).lower()
    
    if response.startswith("yes"):
        return "uses_ai"
    elif "plan to soon" in response:
        return "plans_to_use_ai"
    elif "don't plan" in response:
        return "does_not_plan_to_use_ai"
    else:
        return "other"

stackoverflow_ai_usage_cleaned["ai_adoption_group"] = (
    stackoverflow_ai_usage_cleaned["AISelect"].apply(classify_ai_select)
)

stackoverflow_ai_summary = (
    stackoverflow_ai_usage_cleaned
    .groupby(["year", "ai_adoption_group"])
    .size()
    .reset_index(name="count")
)

year_totals = (
    stackoverflow_ai_usage_cleaned
    .groupby("year")
    .size()
    .reset_index(name="total_respondents")
)

stackoverflow_ai_summary = stackoverflow_ai_summary.merge(year_totals, on="year")
stackoverflow_ai_summary["percentage"] = (
    stackoverflow_ai_summary["count"] / stackoverflow_ai_summary["total_respondents"] * 100
).round(2)

stackoverflow_ai_summary.to_csv(
    PROCESSED_DIR / "stackoverflow_ai_adoption_summary.csv",
    index=False
)

print("Saved file:", PROCESSED_DIR / "stackoverflow_ai_adoption_summary.csv")
stackoverflow_ai_summary

Saved file: /Users/beren/Desktop/dsa/data/processed/stackoverflow_ai_adoption_summary.csv


,year,ai_adoption_group,count,total_respondents,percentage
0,2023,does_not_plan_to_use_ai,26221,89184,29.40
1,2023,missing,1211,89184,1.36
2,2023,plans_to_use_ai,22710,89184,25.46
3,2023,uses_ai,39042,89184,43.78
4,2024,does_not_plan_to_use_ai,14837,65437,22.67
5,2024,missing,4530,65437,6.92
6,2024,plans_to_use_ai,8408,65437,12.85
7,2024,uses_ai,37662,65437,57.55
8,2025,does_not_plan_to_use_ai,5454,49191,11.09
9,2025,missing,15471,49191,31.45


### Valid Response-Based AI Adoption Rate

Some respondents did not answer the AI usage question, so they appear as `missing`. To make the adoption rate easier to interpret, I also calculate the percentage only among respondents who gave a valid answer to the AISelect question.

**Code purpose:** Recalculate AI adoption rates after removing missing responses.

In [83]:
stackoverflow_valid = stackoverflow_ai_usage_cleaned[
    stackoverflow_ai_usage_cleaned["ai_adoption_group"] != "missing"
].copy()

stackoverflow_valid_summary = (
    stackoverflow_valid
    .groupby(["year", "ai_adoption_group"])
    .size()
    .reset_index(name="count")
)

valid_year_totals = (
    stackoverflow_valid
    .groupby("year")
    .size()
    .reset_index(name="valid_responses")
)

stackoverflow_valid_summary = stackoverflow_valid_summary.merge(
    valid_year_totals,
    on="year"
)

stackoverflow_valid_summary["percentage"] = (
    stackoverflow_valid_summary["count"] /
    stackoverflow_valid_summary["valid_responses"] * 100
).round(2)

stackoverflow_valid_summary.to_csv(
    PROCESSED_DIR / "stackoverflow_ai_adoption_valid_summary.csv",
    index=False
)

print("Saved file:", PROCESSED_DIR / "stackoverflow_ai_adoption_valid_summary.csv")
stackoverflow_valid_summary

Saved file: /Users/beren/Desktop/dsa/data/processed/stackoverflow_ai_adoption_valid_summary.csv


,year,ai_adoption_group,count,valid_responses,percentage
0,2023,does_not_plan_to_use_ai,26221,87973,29.81
1,2023,plans_to_use_ai,22710,87973,25.81
2,2023,uses_ai,39042,87973,44.38
3,2024,does_not_plan_to_use_ai,14837,60907,24.36
4,2024,plans_to_use_ai,8408,60907,13.80
5,2024,uses_ai,37662,60907,61.84
6,2025,does_not_plan_to_use_ai,5454,33720,16.17
7,2025,plans_to_use_ai,1797,33720,5.33
8,2025,uses_ai,26469,33720,78.50


### Interpretation of Stack Overflow AI Adoption

After removing missing responses, the share of respondents using AI tools increases from 44.38% in 2023 to 61.84% in 2024 and 78.50% in 2025. This supports the idea that AI tools are becoming more common in online problem-solving and help-seeking contexts.

This survey is not used as the main dataset, but it provides external validation for the Google Trends-based analysis.

## 7. Preparing Pew AI Survey Data

This section prepares the Pew AI survey data as another external validation source. While Google Trends shows search interest, Pew data provides survey-based context about public awareness and attitudes toward AI.

First, I load the raw Pew dataset and inspect its structure before selecting the columns that are relevant to the project.

**Code purpose:** Load the Pew AI survey dataset and inspect its structure.

In [84]:
pew_file = RAW_DIR / "pew_global_ai_survey_2025.csv"

pew_raw = pd.read_csv(pew_file, low_memory=False)

print("Shape:", pew_raw.shape)
pew_raw.head()

Shape: (28333, 483)


,survey,country,weight,id,econ_sit,satisfied_democracy,polsys_satisfied,fav_us,fav_china,fav_eu,...,region_skorea,region_spain,region_sweden,region_turkey,region_uk,urbanicity,qdate_s,qdate_e,intdate_aus,phone_sample
0,1202501,2,0.592005,8001432,2,2,,4,3,2,...,,,,,,,3/23/2025,3/23/2025,,1
1,1202501,2,0.446814,8003472,4,3,,4,4,2,...,,,,,,,3/11/2025,3/11/2025,,1
2,1202501,2,1.120110,8006468,2,1,,3,8,2,...,,,,,,,4/1/2025,4/1/2025,,1
3,1202501,2,0.695317,8006970,2,1,,4,8,1,...,,,,,,,3/10/2025,3/10/2025,,1
4,1202501,2,0.369352,8008601,2,2,,4,3,1,...,,,,,,,3/23/2025,3/23/2025,,1


**Code purpose:** Load the Pew data dictionary to understand the meaning of survey variables.

In [85]:
SCHEMA_DIR = RAW_DIR / "schemas"

pew_dictionary_file = SCHEMA_DIR / "pew_global_ai_survey_2025_dictionary.xlsx"

pew_dictionary = pd.read_excel(pew_dictionary_file, header=1)

print("Dictionary file:", pew_dictionary_file)
print("Dictionary shape:", pew_dictionary.shape)

pew_dictionary.head()

Dictionary file: /Users/beren/Desktop/dsa/data/raw/schemas/pew_global_ai_survey_2025_dictionary.xlsx
Dictionary shape: (483, 9)


,Variable,Position,Label,Measurement Level,Role,Column Width,Alignment,Print Format,Write Format
0,survey,1,Survey,Scale,Input,10,Right,F8,F8
1,country,2,Country,Scale,Input,10,Right,F8,F8
2,weight,3,Weight,Scale,Input,10,Right,F8.2,F8.2
3,id,4,Respondent ID,Scale,Input,10,Right,F2,F2
4,econ_sit,5,Q1. How would you describe the current economi...,Nominal,Input,10,Right,F2,F2


**Code purpose:** Search the Pew dictionary for variables related to artificial intelligence.

In [86]:
ai_related_dictionary = pew_dictionary[
    pew_dictionary["Variable"].astype(str).str.startswith("ai") |
    pew_dictionary["Variable"].astype(str).str.startswith("aireg")
]

print("AI-related dictionary rows:", ai_related_dictionary.shape)

ai_related_dictionary[["Variable", "Label"]]

AI-related dictionary rows: (6, 9)


,Variable,Label
201,ai_heard,"Q30. Artificial intelligence, also known as AI..."
202,ai_cncexc,"Q31. Overall, how would you say the increased ..."
203,aireg_us,Q32a. (SHORTENED) How much trust do you have i...
204,aireg_china,Q32b. (SHORTENED) How much trust do you have i...
205,aireg_eu,Q32c. (SHORTENED) How much trust do you have i...
206,aireg_country,Q32d. (SHORTENED) How much trust do you have i...


**Code purpose:** Select the Pew survey variables related to AI awareness, attitudes, and regulation trust.

In [87]:
pew_ai_variables = [
    "country",
    "weight",
    "ai_heard",
    "ai_cncexc",
    "aireg_us",
    "aireg_china",
    "aireg_eu",
    "aireg_country"
]

pew_ai_survey_cleaned = pew_raw[pew_ai_variables].copy()

pew_ai_survey_cleaned.to_csv(
    PROCESSED_DIR / "pew_ai_survey_cleaned.csv",
    index=False
)

print("Saved file:", PROCESSED_DIR / "pew_ai_survey_cleaned.csv")
print("Shape:", pew_ai_survey_cleaned.shape)

pew_ai_survey_cleaned.head()

Saved file: /Users/beren/Desktop/dsa/data/processed/pew_ai_survey_cleaned.csv
Shape: (28333, 8)


,country,weight,ai_heard,ai_cncexc,aireg_us,aireg_china,aireg_eu,aireg_country
0,2,0.592005,2,2,2,8,2,2
1,2,0.446814,2,3,4,4,2,2
2,2,1.120110,2,2,4,4,8,2
3,2,0.695317,2,2,4,4,1,2
4,2,0.369352,1,3,3,3,2,2


**Code purpose:** Check the distribution of selected Pew AI awareness and attitude variables.

In [88]:
pew_ai_survey_cleaned[["ai_heard", "ai_cncexc"]].value_counts(dropna=False).head(20)

ai_heard  ai_cncexc
2         3            5981
          2            4857
1         3            4277
          2            2709
          1            2345
3         3            2004
2         1            1726
3         2            1557
          8            1067
          1             568
8         8             442
2         8             281
8         3             143
          2             123
          1              67
1         8              63
3         9              44
2         9              33
1         9              13
8         9              10
Name: count, dtype: int64

### Creating a Pew AI Survey Summary

After selecting the AI-related survey columns, I summarize the main Pew AI variables. I focus on AI awareness and general emotional response to AI because these variables provide useful external context for the project.

The response values are kept in their original coded form because the next analysis notebook can use the Pew dictionary to interpret them.

**Code purpose:** Group Pew AI awareness and attitude responses into summary tables and calculate percentages.

In [89]:
pew_ai_heard_summary = (
    pew_ai_survey_cleaned
    .groupby("ai_heard")
    .size()
    .reset_index(name="count")
)

pew_ai_heard_summary["percentage"] = (
    pew_ai_heard_summary["count"] /
    pew_ai_heard_summary["count"].sum() * 100
).round(2)

pew_ai_cncexc_summary = (
    pew_ai_survey_cleaned
    .groupby("ai_cncexc")
    .size()
    .reset_index(name="count")
)

pew_ai_cncexc_summary["percentage"] = (
    pew_ai_cncexc_summary["count"] /
    pew_ai_cncexc_summary["count"].sum() * 100
).round(2)

pew_ai_heard_summary.to_csv(
    PROCESSED_DIR / "pew_ai_heard_summary.csv",
    index=False
)

pew_ai_cncexc_summary.to_csv(
    PROCESSED_DIR / "pew_ai_attitude_summary.csv",
    index=False
)

print("Saved file:", PROCESSED_DIR / "pew_ai_heard_summary.csv")
print("Saved file:", PROCESSED_DIR / "pew_ai_attitude_summary.csv")

print("\nAI awareness summary:")
print(pew_ai_heard_summary)

print("\nAI attitude summary:")
print(pew_ai_cncexc_summary)

Saved file: /Users/beren/Desktop/dsa/data/processed/pew_ai_heard_summary.csv
Saved file: /Users/beren/Desktop/dsa/data/processed/pew_ai_attitude_summary.csv

AI awareness summary:
   ai_heard  count  percentage
0         1   9407       33.20
1         2  12878       45.45
2         3   5240       18.49
3         8    785        2.77
4         9     23        0.08

AI attitude summary:
   ai_cncexc  count  percentage
0          1   4708       16.62
1          2   9252       32.65
2          3  12409       43.80
3          8   1862        6.57
4          9    102        0.36


### Pew Survey Output

The Pew AI survey subset and summary files were saved successfully. The cleaned subset includes variables related to AI awareness, emotional response to AI, and trust in AI regulation. The summary files focus on AI awareness and general attitudes toward AI.

These files will be used as external survey context in the next analysis notebook.

## 8. Final Processed Data Check

This final step checks the processed folder to confirm that all cleaned datasets were created successfully.

**Code purpose:** Confirm that all processed datasets were created successfully.

In [91]:
print("Processed files created:")

for file in PROCESSED_DIR.iterdir():
    if file.name != ".DS_Store":
        print("-", file.name)

Processed files created:
- stackoverflow_ai_adoption_summary.csv
- pew_ai_survey_cleaned.csv
- pew_ai_heard_summary.csv
- stackoverflow_ai_usage_cleaned.csv
- platform_traffic_context.csv
- stackoverflow_ai_adoption_valid_summary.csv
- google_trends_cleaned.csv
- pew_ai_attitude_summary.csv


## Conclusion

In this notebook, the raw datasets were cleaned and reorganized into analysis-ready files. The processed outputs include cleaned Google Trends data, platform traffic context data, Stack Overflow AI adoption summaries, and a Pew AI survey subset.

These files will be used in the next notebook for exploratory data analysis, visualizations, and hypothesis testing.